# Day 21 (Extension): CNN for Image Restoration — Denoising, Deblurring & Super-Resolution

> **Goal:** Extend your CNN skills to image restoration tasks. Build a synthetic degradation pipeline, implement U-Net with skip connections, evaluate using PSNR and SSIM, and compare denoising vs. deblurring performance.

> **Prerequisite:** Day 21 (Build a CNN from Scratch — Complete Project)

---

## Why This Extension Exists

You've built CNNs for classification. Now let's answer: **Can CNNs restore degraded images?**

The answer is **yes** — and the architectures are almost the same. The key differences:

| Component | Classifier (Day 21) | Restoration (Today) |
|---|---|---|
| **Output** | 10 class probabilities | Degraded image → restored image (same size) |
| **Loss** | CrossEntropy | MSE / L1 (pixel comparison) |
| **Architecture** | Encoder → FC head | Encoder → Decoder (U-Net) with skip connections |
| **Evaluation** | Accuracy (%) | PSNR (dB), SSIM (0–1) |

### Optics Connections

- **Gaussian blur = convolution with a PSF.** You know this from optics — the point spread function (PSF) blurs the image. We're learning to *invert* this PSF.
- **PSNR in dB** is the same unit you use for camera SNR characterization.
- **Denoising** is like a content-aware spatial filter — adapts to edges vs. flat regions.
- **Deblurring** is learned deconvolution — the network figures out the inverse filter.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from scipy.ndimage import gaussian_filter

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Synthetic Degradation Pipeline

We'll create three types of degradation on-the-fly:

1. **Gaussian noise** — additive white noise with controllable σ
2. **Gaussian blur** — convolution with a PSF (Gaussian kernel)
3. **Downscaling** — bicubic downscale 2× (for super-resolution)

All are implemented using `scipy.ndimage.gaussian_filter` and basic tensor operations — no new packages needed.

In [ ]:
def add_gaussian_noise(img, sigma=0.1):
    """Add Gaussian noise to a normalized image tensor.
    
    Args:
        img: Tensor (C, H, W) in range [0, 1]
        sigma: Standard deviation of noise
    Returns:
        Noisy image tensor
    """
    noise = torch.randn_like(img) * sigma
    return torch.clamp(img + noise, 0, 1)


def apply_gaussian_blur(img, kernel_size=5, sigma=1.5):
    """Apply Gaussian blur using scipy's gaussian_filter.
    
    In optics terms: this convolves the image with a PSF (Gaussian point spread function).
    
    Args:
        img: Tensor (C, H, W) in range [0, 1]
        kernel_size: Size of the PSF kernel
        sigma: Standard deviation of the Gaussian PSF
    Returns:
        Blurred image tensor
    """
    # Convert to numpy for scipy
    img_np = img.cpu().numpy()
    blurred = np.zeros_like(img_np)
    for c in range(img_np.shape[0]):
        blurred[c] = gaussian_filter(img_np[c], sigma=sigma, truncate=(kernel_size-1)/(2*sigma))
    return torch.from_numpy(blurred).to(img.device)


def bicubic_downscale(img, factor=2):
    """Downscale by bicubic interpolation, then upscale back.
    
    This simulates a low-resolution image that needs super-resolution.
    """
    C, H, W = img.shape
    # Downscale
    small = F.interpolate(img.unsqueeze(0), scale_factor=1/factor, mode="bicubic", align_corners=False)
    # Upscale back
    restored = F.interpolate(small, size=(H, W), mode="bicubic", align_corners=False)
    return restored.squeeze(0)


# Demonstrate degradation effects
demo_img = torch.rand(3, 32, 32)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(demo_img.permute(1, 2, 0))
axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(add_gaussian_noise(demo_img, sigma=0.2).permute(1, 2, 0))
axes[1].set_title("Noise (σ=0.2)"); axes[1].axis("off")
axes[2].imshow(apply_gaussian_blur(demo_img, sigma=1.5).permute(1, 2, 0))
axes[2].set_title("Blur (σ=1.5)"); axes[2].axis("off")
axes[3].imshow(bicubic_downscale(demo_img).permute(1, 2, 0))
axes[3].set_title("2× Downscale"); axes[3].axis("off")
plt.suptitle("Synthetic Degradation Effects", fontsize=12)
plt.tight_layout()
plt.show()

### Custom Dataset with On-The-Fly Degradation

We create a `DegradedCIFAR10` class that applies degradation in `__getitem__`. This way, the model sees different corruptions each epoch — like data augmentation for restoration.

In [ ]:
class DegradedCIFAR10(Dataset):
    """CIFAR-10 with on-the-fly synthetic degradation.
    
    The dataset returns (degraded_image, clean_image) pairs.
    Degradation types: 'noise', 'blur', 'downscale'
    """
    def __init__(self, base_dataset, degradation_type="noise", sigma=0.1, blur_sigma=1.5, blur_kernel=5):
        self.base = base_dataset
        self.degradation_type = degradation_type
        self.sigma = sigma
        self.blur_sigma = blur_sigma
        self.blur_kernel = blur_kernel

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]  # We don't need labels
        # img is normalized (C, H, W) tensor

        # Denormalize to [0, 1] for degradation
        CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
        CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
        img_01 = img * CIFAR_STD + CIFAR_MEAN
        img_01 = torch.clamp(img_01, 0, 1)

        # Apply degradation
        if self.degradation_type == "noise":
            degraded = add_gaussian_noise(img_01, sigma=self.sigma)
        elif self.degradation_type == "blur":
            degraded = apply_gaussian_blur(img_01, sigma=self.blur_sigma, kernel_size=self.blur_kernel)
        elif self.degradation_type == "downscale":
            degraded = bicubic_downscale(img_01)
        else:
            raise ValueError(f"Unknown degradation: {self.degradation_type}")

        # Re-normalize both degraded and clean
        degraded_norm = (degraded - CIFAR_MEAN.to(degraded.device)) / CIFAR_STD.to(degraded.device)
        return degraded_norm, img

---
## 2. Image Quality Metrics (From Scratch)

Before building models, we need evaluation metrics. We'll implement PSNR and SSIM from scratch — understanding the math first, then we can use library functions.

### PSNR (Peak Signal-to-Noise Ratio)

$$\text{PSNR} = 10 \log_{10} \left( \frac{\text{MAX}^2}{\text{MSE}} \right)$$

- MAX = maximum pixel value (1.0 for normalized images)
- Higher PSNR = better reconstruction. 30 dB+ is generally good, 40 dB+ is excellent
- Directly comparable to dB units used in camera signal-to-noise measurements

### SSIM (Structural Similarity Index)

SSIM compares three components: luminance, contrast, and structure. The formula is:

$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

- SSIM ∈ [0, 1], with 1 = perfect match
- More aligned with human perception than PSNR

In [ ]:
@torch.no_grad()
def compute_psnr(img_pred, img_true, max_val=1.0):
    """Compute PSNR between two image tensors.
    
    Args:
        img_pred: Predicted/reconstructed image (B, C, H, W)
        img_true: Ground truth image (B, C, H, W)
        max_val: Maximum pixel value (1.0 for normalized images)
    Returns:
        PSNR in dB
    """
    mse = F.mse_loss(img_pred, img_true)
    if mse == 0:
        return float('inf')
    psnr = 10 * torch.log10(max_val**2 / mse)
    return psnr.item()


@torch.no_grad()
def compute_ssim(img_pred, img_true, window_size=11, C1=0.01, C2=0.03):
    """Compute SSIM between two image tensors.
    
    Simplified implementation: computes SSIM per image, averages over batch.
    Uses a Gaussian window for local statistics (same as the original paper).
    """
    # Create 1D Gaussian window
    gauss = torch.tensor([
        np.exp(-0.5 * ((x - window_size//2) ** 2) / (1.5 ** 2))
        for x in range(window_size)
    ], dtype=torch.float32)
    gauss = gauss / gauss.sum()

    # Create 2D Gaussian window
    window_2d = gauss[:, None] * gauss[None, :]  # (window, window)
    window_2d = window_2d.expand(img_pred.shape[1], 1, window_size, window_size).to(img_pred.device)

    def gaussian_filter_tensor(x):
        """Apply Gaussian filter via convolution."""
        return F.conv2d(x, window_2d, padding=window_size//2, groups=x.shape[1])

    mu_x = gaussian_filter_tensor(img_pred)
    mu_y = gaussian_filter_tensor(img_true)

    sigma_x_sq = gaussian_filter_tensor(img_pred ** 2) - mu_x ** 2
    sigma_y_sq = gaussian_filter_tensor(img_true ** 2) - mu_y ** 2
    sigma_xy = gaussian_filter_tensor(img_pred * img_true) - mu_x * mu_y

    C1 = C1 ** 2
    C2 = C2 ** 2

    ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / \
               ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x_sq + sigma_y_sq + C2))

    return ssim_map.mean().item()


# Test on random tensors
a = torch.rand(4, 3, 32, 32)
b = a * 0.9 + 0.05  # Slightly different
print(f"PSNR (identical): {compute_psnr(a, a):.2f} dB")
print(f"PSNR (similar):   {compute_psnr(a, b):.2f} dB")
print(f"SSIM (identical): {compute_ssim(a, a):.4f}")
print(f"SSIM (similar):   {compute_ssim(a, b):.4f}")

---
## 3. Baseline: Simple CNN Denoiser

This is the simplest possible CNN for denoising: a stack of conv layers with no pooling (we want the output to be the same size as the input), and no skip connections yet.

```
Conv(3→32) → ReLU → Conv(32→64) → ReLU → Conv(64→32) → ReLU → Conv(32→3)
```

Same backbone as Day 17's SimpleCNN, but without pooling and with a decoder instead of a classifier.

In [ ]:
class SimpleDenoiser(nn.Module):
    """Simple CNN for denoising — no skip connections, no pooling."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


simple_denoiser = SimpleDenoiser().to(device)
total = sum(p.numel() for p in simple_denoiser.parameters())
print(simple_denoiser)
print(f"\nTotal parameters: {total:,}")

# Verify output size matches input
x = torch.randn(1, 3, 32, 32).to(device)
out = simple_denoiser(x)
print(f"\nInput: {x.shape} → Output: {out.shape}")

---
## 4. U-Net: Encoder-Decoder with Skip Connections

U-Net is the canonical architecture for image-to-image tasks. Key innovation: **skip connections** bridge each encoder layer to its corresponding decoder layer.

```
Encoder:  Conv → Conv → Pool → Conv → Conv → Pool → Conv → Conv
              ↓                          ↓                    ↓
           skip 1                      skip 2              skip 3
              ↓                          ↓                    ↓
Decoder:  UpConv(+skip1) → Conv → UpConv(+skip2) → Conv → Conv → Conv
```

**Why skip connections matter for restoration:**
- The encoder compresses spatial info (like Day 17 CNN) — fine details are lost in pooling
- Skip connections **directly pass fine details** from encoder to decoder
- The decoder uses both the compressed semantic info AND the fine-grained spatial info

This is exactly the ResNet skip connection concept from Day 18, applied in a different way!

In [ ]:
class UNet(nn.Module):
    """U-Net for image restoration.
    
    Architecture:
    - Encoder: Conv → BN → ReLU blocks with MaxPool downsampling
    - Decoder: UpConv (ConvTranspose2d) blocks with skip connections
    - Skip connections bridge each encoder level to the corresponding decoder level
    """

    def __init__(self, in_channels=3, out_channels=3, base_channels=64):
        super().__init__()

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )  # → (base, 32, 32)
        self.pool1 = nn.MaxPool2d(2)  # → (base, 16, 16)

        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
        )  # → (base*2, 16, 16)
        self.pool2 = nn.MaxPool2d(2)  # → (base*2, 8, 8)

        self.enc3 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 4, base_channels * 4, 3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
        )  # → (base*4, 8, 8)
        self.pool3 = nn.MaxPool2d(2)  # → (base*4, 4, 4)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, padding=1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 8, base_channels * 4, 3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
        )  # → (base*4, 4, 4)

        # Decoder
        self.up3 = nn.ConvTranspose2d(base_channels * 4, base_channels * 4, 2, stride=2)  # → (base*4, 8, 8)
        self.dec3 = nn.Sequential(
            nn.Conv2d(base_channels * 8, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
        )  # → (base*2, 8, 8)

        self.up2 = nn.ConvTranspose2d(base_channels * 2, base_channels * 2, 2, stride=2)  # → (base*2, 16, 16)
        self.dec2 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )  # → (base, 16, 16)

        self.up1 = nn.ConvTranspose2d(base_channels, base_channels, 2, stride=2)  # → (base, 32, 32)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels // 2, 3, padding=1),
            nn.BatchNorm2d(base_channels // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels // 2, out_channels, 3, padding=1),
        )  # → (out_channels, 32, 32)

    def forward(self, x):
        # Encoder with skip connections
        s1 = self.enc1(x)      # (base, 32, 32)
        p1 = self.pool1(s1)    # (base, 16, 16)

        s2 = self.enc2(p1)     # (base*2, 16, 16)
        p2 = self.pool2(s2)    # (base*2, 8, 8)

        s3 = self.enc3(p2)     # (base*4, 8, 8)
        p3 = self.pool3(s3)    # (base*4, 4, 4)

        b = self.bottleneck(p3)  # (base*4, 4, 4)

        # Decoder with skip connections
        d3 = self.up3(b)               # (base*4, 8, 8)
        d3 = torch.cat([d3, s3], dim=1)  # (base*8, 8, 8) — skip connection!
        d3 = self.dec3(d3)             # (base*2, 8, 8)

        d2 = self.up2(d3)              # (base*2, 16, 16)
        d2 = torch.cat([d2, s2], dim=1)  # (base*4, 16, 16) — skip connection!
        d2 = self.dec2(d2)             # (base, 16, 16)

        d1 = self.up1(d2)              # (base, 32, 32)
        d1 = torch.cat([d1, s1], dim=1)  # (base*2, 32, 32) — skip connection!
        out = self.dec1(d1)            # (3, 32, 32)

        return out


unet = UNet(base_channels=64).to(device)
total = sum(p.numel() for p in unet.parameters())
print(f"U-Net total parameters: {total:,}")

# Verify shapes
x = torch.randn(1, 3, 32, 32).to(device)
out = unet(x)
print(f"Input: {x.shape} → Output: {out.shape}")

---
## 5. Training Pipeline

We'll combine everything: data setup, model training, evaluation, and visualization.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Base CIFAR-10 with normalization (same as Day 17)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_data, val_data = random_split(full_train, [45000, 5000], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 64  # Smaller batch for U-Net (more params)

train_dataset = DegradedCIFAR10(train_data, degradation_type="noise", sigma=0.2)
val_dataset   = DegradedCIFAR10(val_data,   degradation_type="noise", sigma=0.2)
test_dataset  = DegradedCIFAR10(test_data,  degradation_type="noise", sigma=0.2)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Visualize a batch of degraded images
degraded, clean = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    d = (degraded[i] * torch.tensor(CIFAR_STD).view(3,1,1) + torch.tensor(CIFAR_MEAN).view(3,1,1)).clamp(0,1)
    c = (clean[i] * torch.tensor(CIFAR_STD).view(3,1,1) + torch.tensor(CIFAR_MEAN).view(3,1,1)).clamp(0,1)
    axes[0, i].imshow(d.permute(1,2,0))
    axes[0, i].set_title(f"Degraded (σ=0.2)"); axes[0, i].axis("off")
    axes[1, i].imshow(c.permute(1,2,0))
    axes[1, i].set_title("Clean"); axes[1, i].axis("off")
plt.suptitle("Training Data: Degraded vs Clean Pairs")
plt.tight_layout()
plt.show()

### Training and Evaluation Functions

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    total = 0
    for degraded, clean in loader:
        degraded, clean = degraded.to(device), clean.to(device)
        output = model(degraded)
        loss = loss_fn(output, clean)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * degraded.size(0)
        total += degraded.size(0)
    return total_loss / total

@torch.no_grad()
def evaluate_restoration(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    total_psnr = 0
    total_ssim = 0
    total = 0
    for degraded, clean in loader:
        degraded, clean = degraded.to(device), clean.to(device)
        output = model(degraded)
        loss = loss_fn(output, clean)
        total_loss += loss.item() * degraded.size(0)
        total_psnr += compute_psnr(output, clean) * degraded.size(0)
        total_ssim += compute_ssim(output, clean) * degraded.size(0)
        total += degraded.size(0)
    return total_loss / total, total_psnr / total, total_ssim / total

### Train Simple Denoiser (Baseline)

In [ ]:
print("=" * 50)
print("Training Simple Denoiser (Baseline)")
print("=" * 50)

model_simple = SimpleDenoiser().to(device)
loss_fn = nn.MSELoss()
optimizer_simple = optim.Adam(model_simple.parameters(), lr=1e-3)
scheduler_simple = optim.lr_scheduler.CosineAnnealingLR(optimizer_simple, T_max=30)

history_simple = {"train_loss": [], "val_loss": [], "psnr": [], "ssim": []}

print(f"{'Ep':>3} | {'TrLoss':>8} | {'VLoss':>8} | {'PSNR':>6} | {'SSIM':>6} | {'Time':>5}")
print("-" * 50)

for epoch in range(30):
    t0 = time.time()
    tl = train_epoch(model_simple, train_loader, loss_fn, optimizer_simple, device)
    vl, psnr, ssim = evaluate_restoration(model_simple, val_loader, loss_fn, device)
    scheduler_simple.step()
    elapsed = time.time() - t0

    history_simple["train_loss"].append(tl)
    history_simple["val_loss"].append(vl)
    history_simple["psnr"].append(psnr)
    history_simple["ssim"].append(ssim)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:3d} | {tl:8.4f} | {vl:8.4f} | {psnr:5.2f} | {ssim:5.4f} | {elapsed:4.1f}s")

print("\n✅ Simple Denoiser trained!")

### Train U-Net

In [ ]:
print("=" * 50)
print("Training U-Net Denoiser")
print("=" * 50)

model_unet = UNet(base_channels=64).to(device)
optimizer_unet = optim.Adam(model_unet.parameters(), lr=1e-3)
scheduler_unet = optim.lr_scheduler.CosineAnnealingLR(optimizer_unet, T_max=30)
best_psnr = 0

history_unet = {"train_loss": [], "val_loss": [], "psnr": [], "ssim": []}

print(f"{'Ep':>3} | {'TrLoss':>8} | {'VLoss':>8} | {'PSNR':>6} | {'SSIM':>6} | {'Time':>5}")
print("-" * 50)

for epoch in range(30):
    t0 = time.time()
    tl = train_epoch(model_unet, train_loader, loss_fn, optimizer_unet, device)
    vl, psnr, ssim = evaluate_restoration(model_unet, val_loader, loss_fn, device)
    scheduler_unet.step()
    elapsed = time.time() - t0

    history_unet["train_loss"].append(tl)
    history_unet["val_loss"].append(vl)
    history_unet["psnr"].append(psnr)
    history_unet["ssim"].append(ssim)

    if psnr > best_psnr:
        best_psnr = psnr
        torch.save(model_unet.state_dict(), "best_unet_denoiser.pt")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:3d} | {tl:8.4f} | {vl:8.4f} | {psnr:5.2f} | {ssim:5.4f} | {elapsed:4.1f}s")

model_unet.load_state_dict(torch.load("best_unet_denoiser.pt", map_location=device, weights_only=True))
print(f"\n✅ U-Net trained! Best val PSNR: {best_psnr:.2f} dB")

### Compare Results: Simple CNN vs U-Net

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Loss curves
axes[0, 0].plot(history_simple["train_loss"], label="Simple Train")
axes[0, 0].plot(history_simple["val_loss"], label="Simple Val")
axes[0, 0].plot(history_unet["train_loss"], label="UNet Train")
axes[0, 0].plot(history_unet["val_loss"], label="UNet Val")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("MSE Loss")
axes[0, 0].set_title("Loss Comparison"); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# PSNR curves
axes[0, 1].plot(history_simple["psnr"], label="Simple Denoiser")
axes[0, 1].plot(history_unet["psnr"], label="U-Net")
axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("PSNR (dB)")
axes[0, 1].set_title("PSNR Comparison"); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# SSIM curves
axes[1, 0].plot(history_simple["ssim"], label="Simple Denoiser")
axes[1, 0].plot(history_unet["ssim"], label="U-Net")
axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("SSIM")
axes[1, 0].set_title("SSIM Comparison"); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

# Summary text
axes[1, 1].axis("off")
simple_final_psnr = history_simple["psnr"][-1]
unet_final_psnr = history_unet["psnr"][-1]
simple_final_ssim = history_simple["ssim"][-1]
unet_final_ssim = history_unet["ssim"][-1]
summary = (
    f"Final Comparison (30 epochs):\n\n"
    f"Simple Denoiser:\n"
    f"  PSNR: {simple_final_psnr:.2f} dB\n"
    f"  SSIM: {simple_final_ssim:.4f}\n\n"
    f"U-Net:\n"
    f"  PSNR: {unet_final_psnr:.2f} dB\n"
    f"  SSIM: {unet_final_ssim:.4f}\n\n"
    f"Improvement:\n"
    f"  PSNR: +{unet_final_psnr - simple_final_psnr:.2f} dB\n"
    f"  SSIM: +{unet_final_ssim - simple_final_ssim:.4f}"
)
axes[1, 1].text(0.1, 0.5, summary, fontsize=12, verticalalignment="center")

plt.tight_layout()
plt.show()

### Visual Comparison: Degraded → Restored → Ground Truth

In [ ]:
model_simple.eval()
model_unet.eval()
degraded, clean = next(iter(test_loader))
degraded, clean = degraded.to(device), clean.to(device)

with torch.no_grad():
    out_simple = model_simple(degraded)
    out_unet = model_unet(degraded)

fig, axes = plt.subplots(4, 5, figsize=(12, 9))
for i in range(5):
    def show(img_tensor, ax):
        img = (img_tensor * torch.tensor(CIFAR_STD).view(3,1,1).to(img_tensor.device)
               + torch.tensor(CIFAR_MEAN).view(3,1,1).to(img_tensor.device)).clamp(0,1)
        ax.imshow(img.cpu().permute(1,2,0)); ax.axis("off")

    show(degraded[i], axes[0, i]); axes[0, i].set_title("Degraded")
    show(out_simple[i], axes[1, i]); axes[1, i].set_title("Simple CNN")
    show(out_unet[i], axes[2, i]); axes[2, i].set_title("U-Net")
    show(clean[i], axes[3, i]); axes[3, i].set_title("Ground Truth")

plt.suptitle("Denoising Comparison (σ=0.2)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Deblurring with U-Net

Now we apply the **same U-Net architecture** to a different degradation: Gaussian blur (PSF convolution). 

**Key insight:** The architecture doesn't change — only the degradation type. This demonstrates that U-Net is a general-purpose image restoration backbone.

**From an optics perspective:** The network must learn to *invert* the PSF — a learned deconvolution. Unlike Wiener filtering (classic approach), the neural network can adapt to image content.

In [ ]:
# Create deblurring datasets
train_blur = DegradedCIFAR10(train_data, degradation_type="blur", blur_sigma=1.5, blur_kernel=5)
val_blur   = DegradedCIFAR10(val_data,   degradation_type="blur", blur_sigma=1.5, blur_kernel=5)
test_blur  = DegradedCIFAR10(test_data,  degradation_type="blur", blur_sigma=1.5, blur_kernel=5)

train_loader_blur = DataLoader(train_blur, batch_size=BATCH_SIZE, shuffle=True)
val_loader_blur   = DataLoader(val_blur,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_blur  = DataLoader(test_blur,  batch_size=BATCH_SIZE, shuffle=False)

# Visualize blur samples
degraded, clean = next(iter(train_loader_blur))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    d = (degraded[i] * torch.tensor(CIFAR_STD).view(3,1,1) + torch.tensor(CIFAR_MEAN).view(3,1,1)).clamp(0,1)
    c = (clean[i] * torch.tensor(CIFAR_STD).view(3,1,1) + torch.tensor(CIFAR_MEAN).view(3,1,1)).clamp(0,1)
    axes[0, i].imshow(d.permute(1,2,0)); axes[0, i].axis("off"); axes[0, i].set_title("Blurred")
    axes[1, i].imshow(c.permute(1,2,0)); axes[1, i].axis("off"); axes[1, i].set_title("Clean")
plt.suptitle("Deblurring Training Data (PSF σ=1.5)")
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 50)
print("Training U-Net Deblurrer")
print("=" * 50)

model_deblur = UNet(base_channels=64).to(device)
optimizer_deblur = optim.Adam(model_deblur.parameters(), lr=1e-3)
scheduler_deblur = optim.lr_scheduler.CosineAnnealingLR(optimizer_deblur, T_max=30)
best_psnr_blur = 0

history_deblur = {"train_loss": [], "val_loss": [], "psnr": [], "ssim": []}

print(f"{'Ep':>3} | {'TrLoss':>8} | {'VLoss':>8} | {'PSNR':>6} | {'SSIM':>6} | {'Time':>5}")
print("-" * 50)

for epoch in range(30):
    t0 = time.time()
    tl = train_epoch(model_deblur, train_loader_blur, loss_fn, optimizer_deblur, device)
    vl, psnr, ssim = evaluate_restoration(model_deblur, val_loader_blur, loss_fn, device)
    scheduler_deblur.step()
    elapsed = time.time() - t0

    history_deblur["train_loss"].append(tl)
    history_deblur["val_loss"].append(vl)
    history_deblur["psnr"].append(psnr)
    history_deblur["ssim"].append(ssim)

    if psnr > best_psnr_blur:
        best_psnr_blur = psnr
        torch.save(model_deblur.state_dict(), "best_unet_deblur.pt")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:3d} | {tl:8.4f} | {vl:8.4f} | {psnr:5.2f} | {ssim:5.4f} | {elapsed:4.1f}s")

model_deblur.load_state_dict(torch.load("best_unet_deblur.pt", map_location=device, weights_only=True))
print(f"\n✅ U-Net Deblurrer trained! Best val PSNR: {best_psnr_blur:.2f} dB")

### Compare Denoising vs Deblurring Results

In [ ]:
print(f"Denoising  — Final PSNR: {history_unet['psnr'][-1]:.2f} dB, SSIM: {history_unet['ssim'][-1]:.4f}")
print(f"Deblurring — Final PSNR: {history_deblur['psnr'][-1]:.2f} dB, SSIM: {history_deblur['ssim'][-1]:.4f}")
print()
print("📝 Which is harder?")
print("Deblurring is typically harder than denoising because:")
print("1. Blur destroys high-frequency information (edges, texture) irreversibly")
print("2. The model must hallucinate (guess) missing high frequencies")
print("3. Denoising preserves edges — just removes random fluctuations")

In [ ]:
model_deblur.eval()
degraded, clean = next(iter(test_loader_blur))
degraded, clean = degraded.to(device), clean.to(device)

with torch.no_grad():
    restored = model_deblur(degraded)

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for i in range(5):
    def show(img_tensor, ax):
        img = (img_tensor * torch.tensor(CIFAR_STD).view(3,1,1).to(img_tensor.device)
               + torch.tensor(CIFAR_MEAN).view(3,1,1).to(img_tensor.device)).clamp(0,1)
        ax.imshow(img.cpu().permute(1,2,0)); ax.axis("off")

    show(degraded[i], axes[0, i]); axes[0, i].set_title("Blurred")
    show(restored[i], axes[1, i]); axes[1, i].set_title("Restored")
    show(clean[i], axes[2, i]); axes[2, i].set_title("Ground Truth")

plt.suptitle("Deblurring Results (PSF σ=1.5)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 🧪 Exercises

### Exercise 1: Noise Level Sweep
Train the U-Net denoiser at different noise levels (σ=0.1, 0.2, 0.5). How does PSNR change with noise level?

### Exercise 2: Remove Skip Connections
Create a version of U-Net without skip connections (just encoder → bottleneck → decoder). Compare PSNR — how much quality do you lose?

### Exercise 3: L1 Loss vs MSE
Train U-Net with L1Loss instead of MSELoss. Which produces sharper results? Compare visually.

### Exercise 4: Real Image Denoising
Load your own photo (e.g., from your phone), add synthetic noise, and run it through the trained denoiser. Does it generalize to real images?

In [ ]:
# Exercise 1: Your code here

In [ ]:
# Exercise 2: Your code here

In [ ]:
# Exercise 3: Your code here

In [ ]:
# Exercise 4: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Synthetic degradation** | Gaussian noise, blur (PSF convolution), and downscaling — full control over corruption |
| **U-Net** | Encoder-decoder with skip connections — preserves fine details |
| **PSNR** | Peak signal-to-noise ratio in dB — directly from MSE, familiar from camera SNR |
| **SSIM** | Structural similarity — luminance, contrast, structure — more aligned with human perception |
| **Skip connections** | Bridge encoder and decoder layers — prevent loss of high-frequency details |
| **Deblurring > Denoising** | Blur destroys frequencies irreversibly; denoising preserves them |

**Next up (Notebook 3):** We'll solve the same problems with Vision Transformers — and compare CNN vs ViT vs Hybrid approaches!